# Cassie 雙足機器人走路訓練
SAC + PyBullet + Stable Baselines3

**流程：** 依序執行所有 Cell → 訓練完自動下載模型
→ 本機跑 `python train_cassie.py --render`

In [ ]:
# Cell 1：安裝套件
!pip install -q pybullet stable-baselines3[extra] gymnasium robot_descriptions 'numpy<2.0'
print('packages installed')

In [ ]:
# Cell 2：掛載 Google Drive（選用 — 斷線後 checkpoint 不會消失）
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/cassie_rl/models/cassie'
    LOG_DIR  = '/content/drive/MyDrive/cassie_rl/logs/cassie'
else:
    SAVE_DIR = '/content/models/cassie'
    LOG_DIR  = '/content/logs/cassie'

import os
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR,  exist_ok=True)
print(f'Save dir: {SAVE_DIR}')

In [ ]:
# Cell 3：Clone repo
import os, sys
REPO_URL = 'https://github.com/DioWang67/pybullet_train.git'
REPO_DIR = '/content/pybullet_train'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('repo ready, cwd:', os.getcwd())

In [ ]:
# Cell 4：環境檢查
from stable_baselines3.common.env_checker import check_env
from envs.cassie_env import CassieEnv

env = CassieEnv()
check_env(env, warn=True)
obs, _ = env.reset()
print('obs shape:', obs.shape)     # (41,)
print('action shape:', env.action_space.shape)  # (10,)
env.close()
print('environment OK')

In [ ]:
# Cell 5：訓練設定
TOTAL_TIMESTEPS = 5_000_000
N_ENVS          = 8
MODEL_PATH      = os.path.join(SAVE_DIR, 'cassie_sac')
VECNORM_PATH    = os.path.join(SAVE_DIR, 'cassie_vecnorm.pkl')

print(f'Timesteps : {TOTAL_TIMESTEPS:,}')
print(f'Envs      : {N_ENVS}')
print(f'Model path: {MODEL_PATH}.zip')

In [ ]:
# Cell 6：訓練
import time, numpy as np
from collections import deque
from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, EvalCallback
from envs.cassie_env import CassieEnv

class SyncVecNormCallback(BaseCallback):
    def __init__(self, train_env, eval_env):
        super().__init__(0)
        self.train_env = train_env
        self.eval_env  = eval_env
    def _on_step(self):
        self.eval_env.obs_rms = self.train_env.obs_rms
        self.eval_env.ret_rms = self.train_env.ret_rms
        return True

class ProgressCallback(BaseCallback):
    FREQ = 10_000
    def __init__(self, total):
        super().__init__(0)
        self.total = total
        self._r = deque(maxlen=20)
        self._best = -np.inf
        self._n = 0
        self._t0 = 0.0
        self._last = 0
    def _on_training_start(self):
        self._t0 = time.time()
        print(f"{'Steps':>12}  {'Eps':>6}  {'MeanR(20)':>10}  {'BestR':>8}  {'FPS':>5}  {'ETA':>9}")
        print('-'*62)
    def _on_step(self):
        for info in self.locals.get('infos', []):
            ep = info.get('episode')
            if ep:
                self._r.append(ep['r'])
                self._n += 1
                if ep['r'] > self._best: self._best = ep['r']
        if self.num_timesteps - self._last >= self.FREQ:
            self._last = self.num_timesteps
            el  = time.time() - self._t0
            fps = int(self.num_timesteps / el) if el > 0 else 0
            rem = (self.total - self.num_timesteps) / fps if fps > 0 else 0
            mr  = np.mean(self._r) if self._r else float('nan')
            m, s = divmod(int(rem), 60); h, m = divmod(m, 60)
            print(f"{self.num_timesteps:>12,}  {self._n:>6}  {mr:>10.1f}  "
                  f"{self._best:>8.1f}  {fps:>5}  {h:02d}:{m:02d}:{s:02d}")
        return True
    def _on_training_end(self):
        el = time.time() - self._t0
        m, s = divmod(int(el), 60); h, m = divmod(m, 60)
        print('-'*62)
        print(f'Done! {h:02d}:{m:02d}:{s:02d}  Best={self._best:.1f}  Eps={self._n}')

make_env = lambda: Monitor(CassieEnv())
vec_env  = DummyVecEnv([make_env] * N_ENVS)
vec_env  = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)
eval_vec = DummyVecEnv([make_env])
eval_vec = VecNormalize(eval_vec, norm_obs=True, norm_reward=False, clip_obs=10.0, training=False)

model = SAC(
    'MlpPolicy', vec_env,
    learning_rate=3e-4, buffer_size=500_000, batch_size=256,
    tau=0.005, gamma=0.99, train_freq=1, gradient_steps=1,
    policy_kwargs={'net_arch': [400, 300]},
    verbose=0, device='auto',
)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[
        SyncVecNormCallback(vec_env, eval_vec),
        ProgressCallback(TOTAL_TIMESTEPS),
        CheckpointCallback(save_freq=200_000, save_path=SAVE_DIR, name_prefix='cassie'),
        EvalCallback(eval_vec, best_model_save_path=SAVE_DIR, log_path=LOG_DIR,
                     eval_freq=100_000, n_eval_episodes=3, deterministic=True, verbose=1),
    ],
)

model.save(MODEL_PATH)
vec_env.save(VECNORM_PATH)
print(f'Model saved -> {MODEL_PATH}.zip')
vec_env.close(); eval_vec.close()

In [ ]:
# Cell 7：打包下載模型
import shutil
from google.colab import files

shutil.make_archive('/content/cassie_model', 'zip', os.path.dirname(SAVE_DIR), os.path.basename(SAVE_DIR))
files.download('/content/cassie_model.zip')
print('下載完成！解壓後放到本機 models/cassie/ 執行 python train_cassie.py --render')

In [ ]:
# Cell 8（選用）：從 checkpoint 繼續訓練
import glob
ckpts = sorted(glob.glob(os.path.join(SAVE_DIR, 'cassie_*.zip')))
if not ckpts:
    print('No checkpoint found.')
else:
    latest = ckpts[-1]
    print(f'Resuming from {latest}')
    make_env = lambda: Monitor(CassieEnv())
    vec_env  = DummyVecEnv([make_env] * N_ENVS)
    vec_env  = VecNormalize.load(VECNORM_PATH, vec_env) if os.path.exists(VECNORM_PATH) \
               else VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)
    vec_env.training = True
    model = SAC.load(latest, env=vec_env, device='auto',
                     custom_objects={'learning_rate': 3e-4, 'lr_schedule': lambda _: 3e-4})
    remaining = TOTAL_TIMESTEPS - model.num_timesteps
    print(f'Steps remaining: {remaining:,}')
    model.learn(total_timesteps=remaining, reset_num_timesteps=False)
    model.save(MODEL_PATH)
    vec_env.save(VECNORM_PATH)
    print('Done.')
    vec_env.close()